# QRT-style mock assessment: data-science coding

**Recommended time: 42 minutes**  
**Libraries:** Python standard library, NumPy, and pandas  
**Deliverable:** complete the three functions and leave the notebook runnable from top to bottom.

This is an original practice problem built from public descriptions of QRT research assessments. It is not an official or leaked QRT question.

## Scenario

You receive a noisy cross-sectional panel for 30 synthetic assets over 160 business days. `signal_raw` is observed at the end of day $t$ and `next_day_return` is the return from $t$ to $t+1$. Your task is to build a clean, leakage-aware out-of-sample evaluation of the signal.

The file deliberately contains duplicate `(date, asset)` observations, missing signals, and signal outliers. Later `observation_time` values are corrections and should win when deduplicating.

### Required conventions

1. Keep the latest `observation_time` for each `(date, asset)` pair.
2. Impute a missing signal with the same-day, same-sector median. If that is unavailable, use the same-day median.
3. On each date, winsorize the imputed signal at the 5th and 95th empirical percentiles.
4. On each date, compute a population z-score (`ddof=0`) called `signal_z`.
5. Evaluate only dates on or after **2025-06-02**.
6. Each day, take an equal-weight long position in the top `floor(n/5)` assets and an equal-weight short position in the bottom `floor(n/5)` assets. Break signal ties by ascending `asset` before taking the tails.
7. The daily factor return is `mean(long next_day_return) - mean(short next_day_return)`.
8. Rank IC is the Pearson correlation between the within-day average ranks of `signal_z` and `next_day_return`.

In [31]:
from pathlib import Path
import math
import numpy as np
import pandas as pd

SEED = 20261028
np.random.seed(SEED)
pd.set_option("display.max_columns", 20)

candidate_paths = [
    Path("data/market_panel.csv"),
    Path("QRT_mock_test/data/market_panel.csv"),
]
DATA_PATH = next((path for path in candidate_paths if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Run this notebook from QRT_mock_test/ or its parent directory.")

raw = pd.read_csv(DATA_PATH)
print(f"Loaded {len(raw):,} rows from {DATA_PATH}")
display(raw.head())

Loaded 4,872 rows from data/market_panel.csv


,date,asset,sector,observation_time,signal_raw,volatility_20d,volume_z,next_day_return
0,2025-06-11 00:00:00,A19,Industrials,2025-06-11 16:00:00,3.115316,0.011724,1.445731,0.016950
1,2025-06-10 00:00:00,A26,Technology,2025-06-10 16:00:00,0.597602,0.010465,-1.376682,-0.008911
2,2025-03-28 00:00:00,A06,Finance,2025-03-28 16:00:00,-0.895440,0.010764,0.037861,-0.006966
3,2025-03-12 00:00:00,A12,Healthcare,2025-03-12 16:00:00,0.798902,0.010565,0.222441,-0.021695
4,2025-04-07 00:00:00,A08,Finance,2025-04-07 16:00:00,-11.899211,0.010875,2.434299,-0.008834


## Task 1 - Clean and standardize the panel

Complete `clean_panel(raw)`. Return one row per `(date, asset)`, sorted by `date` then `asset`, with two new columns:

- `signal_winsor`: the daily winsorized signal;
- `signal_z`: its daily population z-score.

Do not use `next_day_return` while constructing either signal column.

In [32]:
def clean_panel(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.copy()

    # Parse timestamps before sorting.
    df["date"] = pd.to_datetime(df["date"])
    df["observation_time"] = pd.to_datetime(df["observation_time"])

    # Keep the latest correction for each date/asset pair.
    df = df.sort_values(["date", "asset", "observation_time"], kind="mergesort").drop_duplicates(["date", "asset"], keep="last").copy()
    #print(df.head())
    # Impute from the same date and sector, then fall back to the date median.
    sector_day_median = df.groupby(["date", "sector"])["signal_raw"].transform("median")
    #print(df.head())
    day_median = df.groupby("date")["signal_raw"].transform("median")
    #print(df.head())

    df["signal_raw"] = df["signal_raw"].fillna(sector_day_median).fillna(day_median)
    #print(df.head())

    # Compute date-specific winsorization bounds.
    lower = df.groupby("date")["signal_raw"].transform(lambda values: values.quantile(0.05))
    #print(df.head())
    upper = df.groupby("date")["signal_raw"].transform(lambda values: values.quantile(0.95))
    #print(df.head())

    df["signal_winsor"] = df["signal_raw"].clip(lower=lower, upper=upper)
    #print(df.head())

    # Compute daily population z-scores.
    daily_mean = df.groupby("date")["signal_winsor"].transform("mean")
    #print(df.head())
    daily_std = df.groupby("date")["signal_winsor"].transform(lambda values: values.std(ddof=0))
    #print(df.head())

    df["signal_z"] = ((df["signal_winsor"] - daily_mean) / daily_std.replace(0, np.nan)).fillna(0.0)
    #print(df.head())

    return df.sort_values(["date", "asset"]).reset_index(drop=True)

display(clean_panel(raw).head())


,date,asset,sector,observation_time,signal_raw,volatility_20d,volume_z,next_day_return,signal_winsor,signal_z
0,2025-01-02,A00,Energy,2025-01-02 16:00:00,-0.839125,0.010616,0.535232,-0.015641,-0.839125,-1.163559
1,2025-01-02,A01,Energy,2025-01-02 16:00:00,-0.155562,0.010306,0.407572,0.016423,-0.155562,-0.358836
2,2025-01-02,A02,Energy,2025-01-02 16:00:00,-0.319724,0.009756,-0.903093,-0.017693,-0.319724,-0.552095
3,2025-01-02,A03,Energy,2025-01-02 16:00:00,1.458513,0.011328,-0.260889,-0.004256,1.458513,1.541330
4,2025-01-02,A04,Energy,2025-01-02 16:00:00,-1.177762,0.010493,-1.545663,-0.015291,-1.177762,-1.562218


In [33]:
def report_check(name, check):
    try:
        check()
        print(f"PASS - {name}")
    except Exception as exc:
        print(f"NOT YET - {name}: {type(exc).__name__}: {exc}")


def check_clean_panel():
    result = clean_panel(raw.copy())
    assert isinstance(result, pd.DataFrame), "return a DataFrame"
    assert len(result) == 4_800, "latest-observation deduplication should leave 4,800 rows"
    assert not result.duplicated(["date", "asset"]).any(), "duplicate keys remain"
    assert {"signal_winsor", "signal_z"}.issubset(result.columns), "missing output columns"
    assert result["signal_z"].notna().all(), "signal_z contains missing values"
    by_day = result.groupby("date")["signal_z"]
    assert np.allclose(by_day.mean().to_numpy(), 0.0, atol=1e-10), "daily means are not zero"
    assert np.allclose(by_day.std(ddof=0).to_numpy(), 1.0, atol=1e-10), "daily std is not one"


report_check("clean_panel", check_clean_panel)

PASS - clean_panel


## Task 2 - Build the holdout portfolio and rank IC

Complete `compute_daily_performance(clean, holdout_start)`. Return a DataFrame sorted by date with these columns:

- `date`
- `factor_return`
- `rank_ic`
- `n_assets`
- `n_per_leg`

The implementation should work for any holdout date and for days with a different number of assets.

In [34]:
def compute_daily_performance(clean: pd.DataFrame, holdout_start: str) -> pd.DataFrame:
    clean_df = clean_panel(clean)
    start_date = pd.Timestamp(holdout_start)
    mask = clean_df['date'] >= start_date
    clean_df = clean_df.loc[mask]
    #display(clean_df.head())
    daily = []
    for date, group in clean_df.groupby('date'):
        group = group.sort_values(['signal_z', 'asset'], kind='mergesort')
        n_assets = len(group)
        n_per_leg = max(1, int(np.floor(n_assets / 5)))
        short_assets = group.head(n_per_leg)
        #display(short_assets)
        long_assets = group.tail(n_per_leg)
        #display(long_assets)
        long_return = long_assets['next_day_return'].mean()
        #display(long_return)
        short_return = short_assets['next_day_return'].mean()
        #display(short_return)
        factor_return = long_return - short_return
        #display(factor_return)
        signal_rank = group['signal_z'].rank(method='average')
        #display(signal_rank)
        return_rank = group['next_day_return'].rank(method='average')
        #display(return_rank)
        rank_ic = signal_rank.corr(return_rank)
        #display(rank_ic)
        daily.append({
            "date": date,
            "factor_return": factor_return,
            "rank_ic": rank_ic,
            "n_assets": n_assets,
            "n_per_leg": n_per_leg,
        })
        #display(daily)
    daily = pd.DataFrame(daily, columns = ['date', 'factor_return', 'rank_ic', 'n_assets', 'n_per_leg'])
    #display(daily.head())



    return daily

compute_daily_performance(raw, "2025-06-02")

,date,factor_return,rank_ic,n_assets,n_per_leg
0,2025-06-02,0.004140,0.134431,30,6
1,2025-06-03,0.005456,0.208278,30,6
2,2025-06-04,0.009386,0.224324,30,6
3,2025-06-05,-0.006954,-0.148659,30,6
4,2025-06-06,0.002891,0.240125,30,6
5,2025-06-09,0.011718,0.275954,30,6
6,2025-06-10,-0.000313,-0.059197,30,6
7,2025-06-11,-0.004205,-0.096807,30,6
8,2025-06-12,0.007935,0.317980,30,6
9,2025-06-13,-0.014513,-0.253951,30,6


In [35]:
def check_daily_performance():
    clean = clean_panel(raw.copy())
    daily = compute_daily_performance(clean, "2025-06-02")
    assert isinstance(daily, pd.DataFrame), "return a DataFrame"
    expected = {"date", "factor_return", "rank_ic", "n_assets", "n_per_leg"}
    assert expected.issubset(daily.columns), "missing output columns"
    assert daily["date"].is_monotonic_increasing, "dates are not sorted"
    assert (pd.to_datetime(daily["date"]) >= pd.Timestamp("2025-06-02")).all(), "pre-holdout rows remain"
    assert daily[["factor_return", "rank_ic"]].notna().all().all(), "missing metrics"
    assert (daily["n_per_leg"] == (daily["n_assets"] // 5).clip(lower=1)).all(), "wrong leg size"


report_check("compute_daily_performance", check_daily_performance)

PASS - compute_daily_performance


## Task 3 - Summarize performance

Complete `summarize_performance(daily)` and return a dictionary with:

- `n_days`
- `annualized_return` = $252\,\bar r$
- `annualized_volatility` = $\sqrt{252}\,s_r$, with sample standard deviation (`ddof=1`)
- `sharpe_ratio` = annualized return divided by annualized volatility, assuming zero risk-free rate
- `mean_rank_ic`
- `positive_day_rate`
- `max_drawdown`, computed from the compounded equity curve $(1+r_t)$

In [36]:
def summarize_performance(daily: pd.DataFrame) -> dict[str, float]:
    df = daily.copy()
    n_days = len(df)
    returns = df['factor_return']
    an_return = 252 * returns.mean()
    an_vol = np.sqrt(252) * returns.std(ddof=1)
    if an_vol > 0:
         sharpe = an_return / an_vol
    else:
         sharpe = np.nan
    mean_rank_ic = df['rank_ic'].mean()
    positive_day_rate = (returns > 0).mean()
    # Construct the compounded equity curve.
    equity_curve = (1 + returns).cumprod()
    # Highest equity value observed up to each date.
    running_peak = equity_curve.cummax()
    # Percentage decline from the previous running peak.
    drawdown = equity_curve / running_peak - 1
    # Most negative drawdown over the evaluation period.
    max_drawdown = drawdown.min()
    return {
        "n_days": n_days,
        "annualized_return": an_return,
        "annualized_volatility": an_vol,
        "sharpe_ratio": sharpe,
        "mean_rank_ic": mean_rank_ic,
        "positive_day_rate": positive_day_rate,
        "max_drawdown": max_drawdown,
    }
summarize_performance(compute_daily_performance(raw, "2025-06-02"))

{'n_days': 53,
 'annualized_return': np.float64(0.44321325735849054),
 'annualized_volatility': np.float64(0.10134467232184564),
 'sharpe_ratio': np.float64(4.373325673706406),
 'mean_rank_ic': np.float64(0.07807862252445744),
 'positive_day_rate': np.float64(0.660377358490566),
 'max_drawdown': np.float64(-0.019380529788853518)}

In [37]:
def check_summary():
    clean = clean_panel(raw.copy())
    daily = compute_daily_performance(clean, "2025-06-02")
    summary = summarize_performance(daily)
    expected = {
        "n_days", "annualized_return", "annualized_volatility", "sharpe_ratio",
        "mean_rank_ic", "positive_day_rate", "max_drawdown",
    }
    assert expected == set(summary), "return exactly the requested keys"
    assert all(np.isfinite(float(value)) for value in summary.values()), "non-finite metric"
    assert int(summary["n_days"]) == len(daily), "n_days mismatch"
    assert 0.0 <= summary["positive_day_rate"] <= 1.0, "invalid hit rate"
    assert summary["max_drawdown"] <= 0.0, "drawdown must be non-positive"


report_check("summarize_performance", check_summary)

PASS - summarize_performance


## Final run and two-sentence interpretation

When all checks pass, run the cell below. Then replace the prompt after it with exactly two sentences: one on whether the synthetic signal appears useful out of sample, and one caveat about what this backtest does **not** establish.

In [38]:
try:
    clean = clean_panel(raw.copy())
    daily = compute_daily_performance(clean, "2025-06-02")
    summary = summarize_performance(daily)
    display(pd.Series(summary, name="holdout_summary").to_frame())
except Exception as exc:
    print(f"Complete Tasks 1-3, then rerun: {type(exc).__name__}: {exc}")

,holdout_summary
n_days,53.000000
annualized_return,0.443213
annualized_volatility,0.101345
sharpe_ratio,4.373326
mean_rank_ic,0.078079
positive_day_rate,0.660377
max_drawdown,-0.019381


**Your two-sentence interpretation:**  
TODO.

## Submission checklist

- Restart the kernel and run all cells.
- Confirm the three validation lines say `PASS`.
- Keep your code readable and deterministic.
- Do not include the separate answer key in a timed attempt.